1. Load dataset & Keep final test set

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

df = pd.read_json("../donnees/articles.json")

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


: 

2. Development Data

In [ ]:
print("=== DEVELOPMENT DATA ===")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumber of samples:", len(X_train))

print("\nCategory distribution:")
print(y_train.value_counts())

=== DEVELOPMENT DATA ===
X_train shape: (16800,)
y_train shape: (16800,)

Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


3. Preprocessing

In [ ]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

4. Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


In [ ]:
test_text = X_train.iloc[0]

embedding = embedding_model.encode(
    test_text
)

print("Embedding shape:", embedding.shape)
print("First 10 values:")
print(embedding[:10])

Embedding shape: (384,)
First 10 values:
[ 0.23330696  0.23346019 -0.08646365  0.08206552  0.19714399  0.04588521
  0.1144902   0.02201608  0.06691489  0.0040064 ]


In [ ]:
test_embeddings = embedding_model.encode(
    X_train.iloc[:5].tolist()
)

print("Embeddings shape:", test_embeddings.shape)

Embeddings shape: (5, 384)


In [ ]:
X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)
print(X_train_embeddings.shape)

Batches:   0%|          | 0/525 [00:00<?, ?it/s]

(16800, 384)


In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Number of folds:", skf.get_n_splits())

Number of folds: 5


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

embedding_fold_scores = []

for fold, (train_index, val_index) in enumerate(
    skf.split(X_train_embeddings, y_train),
    start=1
):

    # Split embeddings
    X_fold_train = X_train_embeddings[train_index]
    X_fold_val = X_train_embeddings[val_index]

    # Split labels
    y_fold_train = y_train.iloc[train_index]
    y_fold_val = y_train.iloc[val_index]

    # Logistic Regression
    lr_model = LogisticRegression(
        C=1.0,
        max_iter=1000
    )

    lr_model.fit(
        X_fold_train,
        y_fold_train
    )

    # Prediction
    y_fold_pred = lr_model.predict(
        X_fold_val
    )

    # Macro F1
    fold_f1 = f1_score(
        y_fold_val,
        y_fold_pred,
        average="macro"
    )

    embedding_fold_scores.append(fold_f1)

    print(
        f"Fold {fold}: "
        f"Macro F1 = {fold_f1:.4f}"
    )

Fold 1: Macro F1 = 0.8266
Fold 2: Macro F1 = 0.8328
Fold 3: Macro F1 = 0.8414
Fold 4: Macro F1 = 0.8301
Fold 5: Macro F1 = 0.8327


In [ ]:
print("\nEmbedding + Logistic Regression")

print(
    f"Mean Macro F1: "
    f"{np.mean(embedding_fold_scores):.4f}"
)

print(
    f"Standard deviation: "
    f"{np.std(embedding_fold_scores):.4f}"
)


Embedding + Logistic Regression
Mean Macro F1: 0.8327
Standard deviation: 0.0049
